# Ultimatum Game Steering Analysis
## LLM-vs-LLM and LLM-vs-Rule-Based: A Comprehensive Statistical Report

This notebook provides a full empirical analysis of activation-steered ultimatum game experiments.
We compare two evaluation paradigms:

- **LLM-vs-LLM**: both proposer and responder are steered or unsteered Qwen 7B agents
- **LLM-vs-RuleBased** *(7B only)*: proposer or responder is a Qwen 7B LLM; the counterpart follows a fixed rule-based threshold policy

**Structure:** Data loading → per-experiment summaries → statistical tests → effect decomposition → top-config leaderboards → cross-paradigm comparison

> Payoff is a *compound* outcome: `E[payoff] = accept_rate × mean_offer_pct`. Steering can simultaneously shift both components in opposing directions, so we decompose effects throughout.

---
## 0 · Setup

In [ ]:
import json, glob, os, warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── publication-quality style ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'legend.fontsize':   9,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
    'savefig.dpi':       200,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
})

PALETTE = sns.color_palette('tab10')
DIVERGING = sns.color_palette('RdBu_r', 12)

ROOT = Path('/cs/student/projects3/2022/dsurrao/comp0087_snlp_cwk/results/ultimatum')
LVL_DIR  = ROOT / 'llm_vs_llm'
LVRB_DIR = ROOT / 'llm_vs_rulebased'

print('Root exists:', ROOT.exists())
print('LvL experiments:   ', [p.name for p in LVL_DIR.iterdir() if p.is_dir()])
print('LvRB 7B experiments:', [p.name for p in LVRB_DIR.iterdir() if p.is_dir() and '7b' in p.name.lower()])

---
## 1 · Data Loading

In [ ]:
def load_final_best(path: Path) -> dict | None:
    """Load a final_best.json and flatten it into a record dict."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None

    parts = path.parts
    # navigate up to find experiment, layer, role, dimension in path
    dim   = parts[-2]          # e.g. 'firmness'
    role  = parts[-3]          # e.g. 'proposer'
    layer = parts[-4]          # e.g. 'L10'
    exp   = parts[-5]          # experiment directory name

    bl  = d.get('baseline_summary', {})
    fin = d.get('final_summary',    {})
    fs  = d.get('final_stats',      {})

    offer_shift   = fs.get('offer_shift',  {})
    accept_shift  = fs.get('acceptance_shift', {})
    payoff_ref    = fs.get('payoff_delta_reference_only', {})

    # also grab per-alpha detail
    scores = d.get('scores_per_alpha', {})

    rec = dict(
        experiment   = exp,
        layer        = layer,
        layer_num    = int(layer.lstrip('Ll')),
        role         = role,
        dimension    = dim,
        model        = d.get('model', 'unknown'),
        best_alpha   = d.get('best_alpha'),
        best_fitness = d.get('best_fitness'),
        # baseline
        bl_accept_rate       = bl.get('accept_rate'),
        bl_mean_proposer_pct = bl.get('mean_proposer_pct'),
        bl_mean_proposer_payoff = bl.get('mean_proposer_payoff_pct'),
        bl_mean_responder_payoff = bl.get('mean_responder_payoff_pct'),
        bl_n_games           = bl.get('n_games'),
        bl_n_valid           = bl.get('n_valid'),
        bl_n_parse_errors    = bl.get('n_parse_errors', 0),
        # final (best alpha)
        fin_accept_rate       = fin.get('accept_rate'),
        fin_mean_proposer_pct = fin.get('mean_proposer_pct'),
        fin_mean_proposer_payoff = fin.get('mean_proposer_payoff_pct'),
        fin_mean_responder_payoff = fin.get('mean_responder_payoff_pct'),
        fin_n_valid          = fin.get('n_valid'),
        fin_n_parse_errors   = fin.get('n_parse_errors', 0),
        # deltas (final − baseline)
        delta_accept_rate       = (fin.get('accept_rate', np.nan) or np.nan) - (bl.get('accept_rate', np.nan) or np.nan),
        delta_proposer_pct      = (fin.get('mean_proposer_pct', np.nan) or np.nan) - (bl.get('mean_proposer_pct', np.nan) or np.nan),
        delta_proposer_payoff   = (fin.get('mean_proposer_payoff_pct', np.nan) or np.nan) - (bl.get('mean_proposer_payoff_pct', np.nan) or np.nan),
        delta_responder_payoff  = (fin.get('mean_responder_payoff_pct', np.nan) or np.nan) - (bl.get('mean_responder_payoff_pct', np.nan) or np.nan),
        # statistical tests (from final_stats)
        offer_p          = offer_shift.get('p_value'),
        offer_cohens_d   = offer_shift.get('cohens_d'),
        offer_sig        = offer_shift.get('significant_at_0_05', False),
        accept_p         = accept_shift.get('p_value'),
        accept_sig       = accept_shift.get('significant_at_0_05', False),
        payoff_p         = payoff_ref.get('p_value'),
        payoff_cohens_d  = payoff_ref.get('cohens_d'),
        # alpha-level stats (for dose-response)
        scores_per_alpha = scores,
    )
    # compound payoff = accept_rate * proposer_pct / 100  (expected % of pool)
    rec['bl_compound_payoff']  = (rec['bl_accept_rate']  or np.nan) * (rec['bl_mean_proposer_pct']  or np.nan) / 100
    rec['fin_compound_payoff'] = (rec['fin_accept_rate'] or np.nan) * (rec['fin_mean_proposer_pct'] or np.nan) / 100
    rec['delta_compound_payoff'] = rec['fin_compound_payoff'] - rec['bl_compound_payoff']
    return rec


def load_games(path: Path) -> pd.DataFrame | None:
    """Load a games_*.json and return a tidy DataFrame."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None
    games = d.get('games', [])
    if not games:
        return None
    df = pd.DataFrame(games)
    parts = path.parts
    df['dimension'] = parts[-2]
    df['role']      = parts[-3]
    df['layer']     = parts[-4]
    df['experiment']= parts[-5]
    df['alpha_tag'] = d.get('tag', path.stem.replace('games_', ''))
    df['alpha_val'] = d.get('alpha', np.nan)
    df['proposer_pct'] = df['proposer_share'] / df['pool'] * 100
    df['responder_pct_actual'] = df['responder_share'] / df['pool'] * 100
    df['proposer_payoff_pct']  = df['proposer_payoff'] / df['pool'] * 100
    df['responder_payoff_pct'] = df['responder_payoff'] / df['pool'] * 100
    return df

print('Loader functions defined.')

In [ ]:
# ── LLM vs LLM: all experiments ──────────────────────────────────────────
lvl_records = []
lvl_games   = []

for fb_path in sorted(LVL_DIR.rglob('final_best.json')):
    rec = load_final_best(fb_path)
    if rec:
        rec['paradigm'] = 'LLM-vs-LLM'
        lvl_records.append(rec)

for g_path in sorted(LVL_DIR.rglob('games_*.json')):
    gdf = load_games(g_path)
    if gdf is not None:
        gdf['paradigm'] = 'LLM-vs-LLM'
        lvl_games.append(gdf)

# ── LLM vs Rule-Based: 7B only ────────────────────────────────────────────
lvrb_records = []
lvrb_games   = []

for exp_dir in LVRB_DIR.iterdir():
    if not exp_dir.is_dir() or '7b' not in exp_dir.name.lower():
        continue
    for fb_path in sorted(exp_dir.rglob('final_best.json')):
        rec = load_final_best(fb_path)
        if rec:
            rec['paradigm'] = 'LLM-vs-RuleBased'
            lvrb_records.append(rec)
    for g_path in sorted(exp_dir.rglob('games_*.json')):
        gdf = load_games(g_path)
        if gdf is not None:
            gdf['paradigm'] = 'LLM-vs-RuleBased'
            lvrb_games.append(gdf)

# ── Merge ─────────────────────────────────────────────────────────────────
df_sum  = pd.DataFrame(lvl_records + lvrb_records)
df_games = pd.concat(lvl_games + lvrb_games, ignore_index=True)

# drop scores_per_alpha (nested dict, not needed in main table)
scores_col = df_sum.pop('scores_per_alpha')

print(f'Summary records  : {len(df_sum):,}  ({df_sum["paradigm"].value_counts().to_dict()})')
print(f'Game records     : {len(df_games):,}')
print(f'Dimensions       : {sorted(df_sum["dimension"].unique())}')
print(f'Layers           : {sorted(df_sum["layer_num"].unique())}')
print(f'Roles            : {sorted(df_sum["role"].unique())}')

---
## 2 · Dataset Overview

In [ ]:
# Coverage table: how many configs per paradigm × role × layer
cov = (
    df_sum.groupby(['paradigm', 'role', 'layer_num'])
    .agg(n_configs=('dimension', 'count'),
         dimensions=('dimension', lambda x: ', '.join(sorted(x.unique()))))
    .reset_index()
)
print('=== Configuration Coverage ===')
display(cov)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) configs per paradigm
cnt = df_sum['paradigm'].value_counts()
axes[0].bar(cnt.index, cnt.values, color=[PALETTE[0], PALETTE[1]], edgecolor='k', linewidth=0.6)
axes[0].set_title('(a) Configs per Paradigm')
axes[0].set_ylabel('# Configurations')
for bar, v in zip(axes[0].patches, cnt.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(v),
                 ha='center', va='bottom', fontsize=9)

# (b) configs per dimension × paradigm
piv = df_sum.groupby(['paradigm', 'dimension']).size().unstack(fill_value=0)
piv.T.plot(kind='bar', ax=axes[1], color=[PALETTE[0], PALETTE[1]], edgecolor='k',
           linewidth=0.5, width=0.7)
axes[1].set_title('(b) Configs per Dimension')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('# Configs')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)

# (c) configs per layer × paradigm
piv2 = df_sum.groupby(['paradigm', 'layer_num']).size().unstack(fill_value=0)
piv2.T.plot(kind='bar', ax=axes[2], color=[PALETTE[0], PALETTE[1]], edgecolor='k',
            linewidth=0.5, width=0.7)
axes[2].set_title('(c) Configs per Layer')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('# Configs')
axes[2].legend(fontsize=8)

plt.suptitle('Dataset Coverage Overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3 · Baseline Behaviour

Before examining steering effects we characterise the unsteered (baseline) offer and acceptance distributions.

In [ ]:
# Baseline games only
bl_games = df_games[df_games['alpha_tag'] == 'baseline'].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Offer % distribution
for i, (par, grp) in enumerate(bl_games.groupby('paradigm')):
    axes[0].hist(grp['proposer_pct'].dropna(), bins=30, alpha=0.6, label=par,
                 color=PALETTE[i], edgecolor='k', linewidth=0.3)
axes[0].axvline(50, color='red', lw=1.2, linestyle='--', label='50% (equal split)')
axes[0].set_title('(a) Baseline: Proposer Offer (%)')
axes[0].set_xlabel('Proposer share of pool (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Accept rate by dimension
bl_acc = bl_games.groupby(['paradigm', 'dimension'])['agreed'].mean().unstack('paradigm') * 100
bl_acc.plot(kind='bar', ax=axes[1], color=[PALETTE[0], PALETTE[1]], edgecolor='k',
            linewidth=0.5, width=0.7)
axes[1].set_title('(b) Baseline Accept Rate by Dimension')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Acceptance rate (%)')
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)

# Pool size distribution (relevant for variable pools)
for i, (par, grp) in enumerate(bl_games.groupby('paradigm')):
    axes[2].hist(grp['pool'].dropna(), bins=25, alpha=0.6, label=par,
                 color=PALETTE[i], edgecolor='k', linewidth=0.3)
axes[2].set_title('(c) Pool Size Distribution')
axes[2].set_xlabel('Pool size ($)')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('Baseline Game Characteristics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
bl_summary = bl_games.groupby('paradigm').agg(
    n_games=('agreed', 'count'),
    accept_rate=('agreed', 'mean'),
    mean_offer_pct=('proposer_pct', 'mean'),
    std_offer_pct=('proposer_pct', 'std'),
    mean_pool=('pool', 'mean'),
    parse_errors=('parse_error', 'sum'),
).round(3)
print('\nBaseline Summary')
display(bl_summary)

---
## 4 · Payoff Decomposition: The Compound Variable

Proposer expected payoff is:
$$\mathbb{E}[\text{payoff}] = \underbrace{\text{accept\_rate}}_{\text{acceptance component}} \times \underbrace{\text{offer\_pct}}_{\text{extraction component}}$$

Steering can move these in opposite directions. We therefore track all three quantities independently.

In [ ]:
# Compute compound payoff per game then aggregate over baseline & steered
df_games['compound_payoff_pct'] = df_games['agreed'] * df_games['proposer_pct']

# Paired delta per (paradigm, experiment, layer, dimension, role, game_id)
# We compare best-alpha games to baseline games
bl_g = df_games[df_games['alpha_tag'] == 'baseline'][['paradigm','experiment','layer','dimension','role','game_id',
                                                       'proposer_pct','agreed','compound_payoff_pct']].copy()
fin_g = df_games[df_games['alpha_tag'] == 'final'][['paradigm','experiment','layer','dimension','role','game_id',
                                                     'proposer_pct','agreed','compound_payoff_pct']].copy()

bl_g  = bl_g.rename(columns={c: 'bl_'+c  for c in ['proposer_pct','agreed','compound_payoff_pct']})
fin_g = fin_g.rename(columns={c: 'fin_'+c for c in ['proposer_pct','agreed','compound_payoff_pct']})

paired = pd.merge(bl_g, fin_g,
                  on=['paradigm','experiment','layer','dimension','role','game_id'],
                  how='inner')
paired['delta_offer']    = paired['fin_proposer_pct'] - paired['bl_proposer_pct']
paired['delta_agreed']   = paired['fin_agreed'].astype(int) - paired['bl_agreed'].astype(int)
paired['delta_compound'] = paired['fin_compound_payoff_pct'] - paired['bl_compound_payoff_pct']

print(f'Paired game records: {len(paired):,}')

# Scatter: delta_offer vs delta_accept_rate per config
agg_paired = paired.groupby(['paradigm','experiment','layer','dimension','role']).agg(
    delta_offer=('delta_offer','mean'),
    delta_accept=('delta_agreed','mean'),
    delta_compound=('delta_compound','mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (par, grp) in enumerate(agg_paired.groupby('paradigm')):
    sc = axes[0].scatter(grp['delta_offer'], grp['delta_accept']*100,
                         c=[PALETTE[i]]*len(grp), label=par, alpha=0.65, s=40, edgecolors='k', linewidths=0.3)

axes[0].axhline(0, color='k', lw=0.8, linestyle='--')
axes[0].axvline(0, color='k', lw=0.8, linestyle='--')
axes[0].set_xlabel('Δ Proposer offer (%)')
axes[0].set_ylabel('Δ Accept rate (pp)')
axes[0].set_title('(a) Offer vs Acceptance Trade-off')
axes[0].legend()
axes[0].text(0.03, 0.97, 'High demand\nLow accept', transform=axes[0].transAxes,
             va='top', fontsize=8, color='grey')
axes[0].text(0.03, 0.03, 'Low demand\nLow accept', transform=axes[0].transAxes,
             va='bottom', fontsize=8, color='grey')
axes[0].text(0.65, 0.03, 'Low demand\nHigh accept', transform=axes[0].transAxes,
             va='bottom', fontsize=8, color='grey')
axes[0].text(0.65, 0.97, 'High demand\nHigh accept', transform=axes[0].transAxes,
             va='top', fontsize=8, color='grey')

# Histogram of compound payoff delta
for i, (par, grp) in enumerate(agg_paired.groupby('paradigm')):
    axes[1].hist(grp['delta_compound'], bins=25, alpha=0.6, label=par, color=PALETTE[i],
                 edgecolor='k', linewidth=0.3)
axes[1].axvline(0, color='red', lw=1.2, linestyle='--', label='no change')
axes[1].set_xlabel('Δ Compound payoff (pp)')
axes[1].set_ylabel('# Configs')
axes[1].set_title('(b) Distribution of Compound Payoff Change')
axes[1].legend()

plt.suptitle('Payoff Decomposition: Offer × Acceptance', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5 · Statistical Significance and Multiple Testing Correction

In [ ]:
# For each config compute:
#  - paired t-test on offer_pct (from raw games)
#  - Fisher's exact on accept/reject counts (from summary)
#  - paired t-test on compound payoff
# We run this on df_sum which has the pre-computed stats, then apply BH correction.

df_stats = df_sum.copy()

# Replace None with NaN
for col in ['offer_p','accept_p','payoff_p','offer_cohens_d','payoff_cohens_d']:
    df_stats[col] = pd.to_numeric(df_stats[col], errors='coerce')

# BH-FDR correction on offer p-values
valid_offer = df_stats['offer_p'].notna()
if valid_offer.sum() > 0:
    _, pvals_corr, _, _ = multipletests(df_stats.loc[valid_offer, 'offer_p'], method='fdr_bh')
    df_stats.loc[valid_offer, 'offer_p_bh'] = pvals_corr
df_stats['offer_sig_bh'] = df_stats.get('offer_p_bh', pd.Series(dtype=float)) < 0.05

# BH-FDR on accept p-values
valid_acc = df_stats['accept_p'].notna()
if valid_acc.sum() > 0:
    _, pvals_acc_corr, _, _ = multipletests(df_stats.loc[valid_acc, 'accept_p'], method='fdr_bh')
    df_stats.loc[valid_acc, 'accept_p_bh'] = pvals_acc_corr
df_stats['accept_sig_bh'] = df_stats.get('accept_p_bh', pd.Series(dtype=float)) < 0.05

sig_counts = pd.DataFrame({
    'Metric': ['Offer shift', 'Accept shift'],
    'Sig (raw)': [
        df_stats['offer_sig'].sum(),
        df_stats['accept_sig'].sum(),
    ],
    'Sig (BH-FDR)': [
        df_stats['offer_sig_bh'].sum(),
        df_stats['accept_sig_bh'].sum(),
    ],
    'Total configs': [len(df_stats)] * 2
})
print('=== Significance Summary (all paradigms combined) ===')
display(sig_counts)

# By paradigm
sig_par = df_stats.groupby('paradigm').agg(
    offer_sig_raw=('offer_sig','sum'),
    offer_sig_bh=('offer_sig_bh','sum'),
    accept_sig_raw=('accept_sig','sum'),
    accept_sig_bh=('accept_sig_bh','sum'),
    total=('dimension','count'),
)
print('\n=== Significance by Paradigm ===')
display(sig_par)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Volcano plot: effect size vs -log10(p) for offer shift
valid = df_stats.dropna(subset=['offer_cohens_d', 'offer_p'])
colors = valid['paradigm'].map({'LLM-vs-LLM': PALETTE[0], 'LLM-vs-RuleBased': PALETTE[1]})
sig_mask = valid['offer_sig_bh'].fillna(False)

axes[0].scatter(valid['offer_cohens_d'], -np.log10(valid['offer_p'].clip(1e-10)),
                c=colors, alpha=0.6, s=35, edgecolors='none')
axes[0].scatter(valid.loc[sig_mask, 'offer_cohens_d'],
                -np.log10(valid.loc[sig_mask, 'offer_p'].clip(1e-10)),
                c=colors[sig_mask], alpha=1.0, s=55, edgecolors='k', linewidths=0.5, marker='*')
axes[0].axhline(-np.log10(0.05), color='red', lw=1, linestyle='--', label='p=0.05 (BH)')
axes[0].axvline(0, color='grey', lw=0.8, linestyle='--')
axes[0].set_xlabel("Cohen's d (offer shift)")
axes[0].set_ylabel('-log₁₀(p-value)')
axes[0].set_title('(a) Volcano Plot: Offer Shift')
handles = [mpatches.Patch(color=PALETTE[0], label='LLM-vs-LLM'),
           mpatches.Patch(color=PALETTE[1], label='LLM-vs-RuleBased')]
axes[0].legend(handles=handles, fontsize=8)

# Cohen's d distribution by dimension
dim_order = sorted(df_stats['dimension'].dropna().unique())
bp_data = [df_stats.loc[df_stats['dimension']==d, 'offer_cohens_d'].dropna().values for d in dim_order]
bp = axes[1].boxplot(bp_data, labels=dim_order, patch_artist=True, notch=False,
                     medianprops=dict(color='k', lw=1.5))
for patch, color in zip(bp['boxes'], sns.color_palette('husl', len(dim_order))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(0, color='red', lw=0.8, linestyle='--')
axes[1].set_xticklabels(dim_order, rotation=45, ha='right')
axes[1].set_ylabel("Cohen's d")
axes[1].set_title("(b) Effect Size Distribution per Dimension")

plt.suptitle('Statistical Significance Overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6 · Top Configurations by Key Metrics

We rank configurations by three criteria:
1. **Accept rate shift** (Δ acceptance rate, pp)
2. **Offer shift** (Δ proposer demand, pp)
3. **Compound payoff shift** (Δ E[payoff], pp)

All metrics are relative to the unsteered baseline. We report the top 10 across all configs.

In [ ]:
def top_table(df, metric_col, n=10, ascending=False, extra_cols=None):
    cols = ['paradigm','experiment','layer','role','dimension','best_alpha',
            'bl_accept_rate','fin_accept_rate','delta_accept_rate',
            'bl_mean_proposer_pct','fin_mean_proposer_pct','delta_proposer_pct',
            'bl_compound_payoff','fin_compound_payoff','delta_compound_payoff',
            'offer_cohens_d','offer_p','accept_p',
            'offer_sig_bh','accept_sig_bh']
    if extra_cols:
        cols += extra_cols
    available = [c for c in cols if c in df.columns]
    t = df[available].dropna(subset=[metric_col]).sort_values(metric_col, ascending=ascending).head(n)
    return t.round(3)

# ── Accept rate shift (largest increase in acceptance) ─────────────────────
print('═'*90)
print('TOP 10 CONFIGS: Largest INCREASE in Accept Rate')
print('═'*90)
top_acc_up = top_table(df_stats, 'delta_accept_rate', n=10, ascending=False)
display(top_acc_up[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_accept_rate','fin_accept_rate','delta_accept_rate',
                     'offer_cohens_d','accept_p','accept_sig_bh']])

print()
print('─'*90)
print('TOP 10 CONFIGS: Largest DECREASE in Accept Rate')
print('─'*90)
top_acc_dn = top_table(df_stats, 'delta_accept_rate', n=10, ascending=True)
display(top_acc_dn[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_accept_rate','fin_accept_rate','delta_accept_rate',
                     'offer_cohens_d','accept_p','accept_sig_bh']])

In [ ]:
print('═'*90)
print('TOP 10 CONFIGS: Largest INCREASE in Proposer Offer')
print('═'*90)
top_off_up = top_table(df_stats, 'delta_proposer_pct', n=10, ascending=False)
display(top_off_up[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_mean_proposer_pct','fin_mean_proposer_pct','delta_proposer_pct',
                     'offer_cohens_d','offer_p','offer_sig_bh']])

print()
print('─'*90)
print('TOP 10 CONFIGS: Largest DECREASE in Proposer Offer (most generous)')
print('─'*90)
top_off_dn = top_table(df_stats, 'delta_proposer_pct', n=10, ascending=True)
display(top_off_dn[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_mean_proposer_pct','fin_mean_proposer_pct','delta_proposer_pct',
                     'offer_cohens_d','offer_p','offer_sig_bh']])

In [ ]:
print('═'*90)
print('TOP 10 CONFIGS: Best Compound Payoff (E[payoff] = accept_rate × offer_pct)')
print('═'*90)
top_cpd_up = top_table(df_stats, 'delta_compound_payoff', n=10, ascending=False)
display(top_cpd_up[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_compound_payoff','fin_compound_payoff','delta_compound_payoff',
                     'delta_accept_rate','delta_proposer_pct',
                     'offer_cohens_d','offer_sig_bh','accept_sig_bh']])

print()
print('─'*90)
print('TOP 10 CONFIGS: Worst Compound Payoff (largest decrease)')
print('─'*90)
top_cpd_dn = top_table(df_stats, 'delta_compound_payoff', n=10, ascending=True)
display(top_cpd_dn[['paradigm','experiment','layer','role','dimension','best_alpha',
                     'bl_compound_payoff','fin_compound_payoff','delta_compound_payoff',
                     'delta_accept_rate','delta_proposer_pct',
                     'offer_cohens_d','offer_sig_bh','accept_sig_bh']])

---
## 7 · Heatmaps: Dimension × Layer Effect Landscape

In [ ]:
def make_heatmap(df, metric, title, ax, cmap='RdBu_r', center=0, fmt='.1f'):
    piv = df.pivot_table(values=metric, index='dimension', columns='layer_num', aggfunc='mean')
    if piv.empty:
        ax.set_visible(False)
        return
    vmax = max(abs(piv.values[~np.isnan(piv.values)]).max(), 0.01)
    sns.heatmap(piv, ax=ax, cmap=cmap, center=center, vmin=-vmax, vmax=vmax,
                annot=True, fmt=fmt, linewidths=0.5, linecolor='white',
                cbar_kws={'shrink': 0.75, 'label': metric})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Dimension')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

for par in ['LLM-vs-LLM', 'LLM-vs-RuleBased']:
    sub = df_stats[df_stats['paradigm'] == par]
    if sub.empty:
        continue
    for role in ['proposer', 'responder']:
        sub_r = sub[sub['role'] == role]
        if sub_r.empty:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(18, max(4, len(sub_r['dimension'].unique())*0.45 + 1)))
        make_heatmap(sub_r, 'delta_accept_rate', 'Δ Accept Rate (pp)', axes[0])
        make_heatmap(sub_r, 'delta_proposer_pct', 'Δ Proposer Offer (%)', axes[1])
        make_heatmap(sub_r, 'delta_compound_payoff', 'Δ Compound Payoff (pp)', axes[2])
        fig.suptitle(f'{par} | Role: {role.capitalize()}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

---
## 8 · Dose-Response: Alpha × Metric Curves

In [ ]:
# Rebuild per-alpha tidy table from games data
alpha_games = df_games[df_games['alpha_tag'] != 'final'].copy()

# Map tags to numeric alpha
def parse_alpha(tag):
    if tag == 'baseline':
        return 0.0
    try:
        return float(str(tag).replace('alpha', ''))
    except:
        return np.nan

alpha_games['alpha_num'] = alpha_games['alpha_val'].fillna(
    alpha_games['alpha_tag'].map(parse_alpha))

dose = (
    alpha_games
    .groupby(['paradigm','experiment','layer','role','dimension','alpha_num'])
    .agg(
        accept_rate=('agreed','mean'),
        mean_offer=('proposer_pct','mean'),
        compound=('compound_payoff_pct','mean'),
        n=('agreed','count'),
    )
    .reset_index()
)

# Plot dose-response for top dimensions by Cohen's d (across both paradigms)
top_dims = (
    df_stats.groupby('dimension')['offer_cohens_d'].apply(lambda x: x.abs().mean())
    .sort_values(ascending=False).head(6).index.tolist()
)

fig, axes = plt.subplots(3, len(top_dims), figsize=(4*len(top_dims), 10))
metrics    = ['accept_rate', 'mean_offer', 'compound']
ylabels    = ['Accept Rate', 'Mean Offer (%)', 'Compound Payoff']
multipliers= [100, 1, 1]

for col, dim in enumerate(top_dims):
    sub = dose[dose['dimension'] == dim]
    for row, (met, yl, mul) in enumerate(zip(metrics, ylabels, multipliers)):
        ax = axes[row, col]
        for i, (par, grp) in enumerate(sub.groupby('paradigm')):
            agg = grp.groupby('alpha_num')[met].mean() * mul
            ax.plot(agg.index, agg.values, 'o-', color=PALETTE[i], label=par,
                    markersize=4, linewidth=1.5)
        ax.axvline(0, color='grey', lw=0.8, linestyle='--')
        if row == 0:
            ax.set_title(dim, fontsize=10, fontweight='bold')
        if col == 0:
            ax.set_ylabel(yl, fontsize=9)
        if row == 2:
            ax.set_xlabel('Alpha')
        if row == 0 and col == 0:
            ax.legend(fontsize=7)

plt.suptitle('Dose-Response: Steering Intensity (α) vs Outcome Metrics\n(Top 6 Dimensions by Mean |Cohen\'s d|)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9 · Role Analysis: Proposer vs Responder Steering

In [ ]:
role_agg = (
    df_stats
    .groupby(['paradigm', 'role'])
    .agg(
        mean_delta_offer     = ('delta_proposer_pct',   'mean'),
        std_delta_offer      = ('delta_proposer_pct',   'std'),
        mean_delta_accept    = ('delta_accept_rate',     'mean'),
        std_delta_accept     = ('delta_accept_rate',     'std'),
        mean_delta_compound  = ('delta_compound_payoff', 'mean'),
        std_delta_compound   = ('delta_compound_payoff', 'std'),
        frac_sig_offer       = ('offer_sig_bh',          'mean'),
        frac_sig_accept      = ('accept_sig_bh',         'mean'),
        n                    = ('dimension',             'count'),
    )
    .round(3)
)
print('=== Role-Level Summary ===')
display(role_agg)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['delta_proposer_pct', 'delta_accept_rate', 'delta_compound_payoff']
titles  = ['Δ Proposer Offer (%)', 'Δ Accept Rate (pp)', 'Δ Compound Payoff (pp)']
scales  = [1, 100, 1]

for ax, met, title, sc in zip(axes, metrics, titles, scales):
    for i, (par, grp) in enumerate(df_stats.groupby('paradigm')):
        for j, role in enumerate(['proposer', 'responder']):
            vals = grp.loc[grp['role'] == role, met].dropna() * sc
            x = i*3 + j
            bp = ax.boxplot(vals, positions=[x], widths=0.6, patch_artist=True,
                            medianprops=dict(color='k', lw=1.5),
                            boxprops=dict(facecolor=PALETTE[i], alpha=0.6))
    ax.axhline(0, color='red', lw=0.8, linestyle='--')
    ax.set_xticks([0.5, 3.5])
    ax.set_xticklabels(['LLM-vs-LLM', 'LLM-vs-RuleBased'])
    ax.set_title(title)
    ax.set_ylabel('Change (best alpha vs baseline)')

# Legend for roles
handles = [mpatches.Patch(color='lightblue', label='proposer'),
           mpatches.Patch(color='lightyellow', label='responder')]
fig.legend(handles=handles, loc='upper right', fontsize=9)
plt.suptitle('Role Comparison: Proposer vs Responder Steering Effects', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: mean delta per (dimension, role) for each paradigm
for par in df_stats['paradigm'].unique():
    sub = df_stats[df_stats['paradigm'] == par]
    piv = sub.pivot_table(values='delta_compound_payoff', index='dimension', columns='role', aggfunc='mean')
    if piv.empty:
        continue
    ax = piv.plot(kind='bar', figsize=(12, 4), color=[PALETTE[2], PALETTE[3]],
                  edgecolor='k', linewidth=0.5, width=0.7)
    plt.axhline(0, color='k', lw=0.8, linestyle='--')
    plt.title(f'{par}: Compound Payoff Δ by Dimension and Role', fontsize=12, fontweight='bold')
    plt.xlabel('Dimension')
    plt.ylabel('Δ Compound Payoff (pp)')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Role')
    plt.tight_layout()
    plt.show()

---
## 10 · Layer Analysis: Where in the Network Does Steering Work?

In [ ]:
layer_agg = (
    df_stats
    .groupby(['paradigm', 'layer_num'])
    .agg(
        mean_abs_cohens_d    = ('offer_cohens_d',       lambda x: x.abs().mean()),
        mean_delta_offer     = ('delta_proposer_pct',   'mean'),
        mean_delta_accept    = ('delta_accept_rate',     'mean'),
        mean_delta_compound  = ('delta_compound_payoff', 'mean'),
        frac_sig             = ('offer_sig_bh',          'mean'),
        n                    = ('dimension',             'count'),
    )
    .reset_index()
    .round(3)
)
print('=== Layer Summary ===')
display(layer_agg)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

metrics = ['mean_abs_cohens_d', 'mean_delta_offer', 'mean_delta_accept', 'mean_delta_compound']
titles  = ["Mean |Cohen's d| (offer)", 'Mean Δ Offer (%)', 'Mean Δ Accept Rate', 'Mean Δ Compound Payoff']

for ax, met, title in zip(axes, metrics, titles):
    for i, (par, grp) in enumerate(layer_agg.groupby('paradigm')):
        grp_s = grp.sort_values('layer_num')
        ax.plot(grp_s['layer_num'], grp_s[met], 'o-', color=PALETTE[i], label=par, linewidth=2, markersize=7)
    ax.axhline(0, color='grey', lw=0.8, linestyle='--')
    ax.set_xlabel('Layer')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle('Effect Magnitude vs Network Layer', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11 · Dimension Deep-Dives

For each dimension, we show the full distribution of outcomes across configs, and the per-layer breakdown.

In [ ]:
dims_all = sorted(df_stats['dimension'].dropna().unique())
n_dims   = len(dims_all)

fig, axes = plt.subplots(3, n_dims, figsize=(2.5*n_dims, 9))
if n_dims == 1:
    axes = axes.reshape(-1, 1)

metrics = ['delta_proposer_pct', 'delta_accept_rate', 'delta_compound_payoff']
ylabels = ['Δ Offer (%)', 'Δ Accept Rate', 'Δ Compound Payoff']

for col, dim in enumerate(dims_all):
    sub = df_stats[df_stats['dimension'] == dim]
    for row, (met, yl) in enumerate(zip(metrics, ylabels)):
        ax = axes[row, col]
        for i, (par, grp) in enumerate(sub.groupby('paradigm')):
            vals = grp[met].dropna()
            if len(vals) == 0:
                continue
            ax.scatter([i]*len(vals), vals, color=PALETTE[i], alpha=0.7, s=20, zorder=3)
            ax.plot([i-0.3, i+0.3], [vals.mean(), vals.mean()], color=PALETTE[i], lw=2)
        ax.axhline(0, color='grey', lw=0.8, linestyle='--')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['LvL', 'LvRB'], fontsize=7)
        if col == 0:
            ax.set_ylabel(yl, fontsize=8)
        if row == 0:
            ax.set_title(dim, fontsize=9, fontweight='bold')

plt.suptitle('Per-Dimension Distribution of Effects\n(dot = config; bar = mean)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed dimension table
dim_table = (
    df_stats
    .groupby(['dimension', 'paradigm', 'role'])
    .agg(
        n_configs=('best_alpha','count'),
        mean_delta_offer=('delta_proposer_pct','mean'),
        mean_delta_accept=('delta_accept_rate','mean'),
        mean_delta_compound=('delta_compound_payoff','mean'),
        mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
        frac_sig_offer=('offer_sig_bh','mean'),
        frac_sig_accept=('accept_sig_bh','mean'),
        best_compound_delta=('delta_compound_payoff','max'),
    )
    .round(3)
    .reset_index()
    .sort_values('mean_delta_compound', ascending=False)
)
print('=== Dimension × Paradigm × Role Summary ===')
display(dim_table)

---
## 12 · Cross-Paradigm Comparison: LLM-vs-LLM vs LLM-vs-RuleBased

In [ ]:
# For configs that appear in both paradigms: match on (dimension, role, layer_num)
lvl_grp  = df_stats[df_stats['paradigm']=='LLM-vs-LLM'].groupby(['dimension','role','layer_num'])
lvrb_grp = df_stats[df_stats['paradigm']=='LLM-vs-RuleBased'].groupby(['dimension','role','layer_num'])

lvl_agg = df_stats[df_stats['paradigm']=='LLM-vs-LLM'].groupby(['dimension','role']).agg(
    mean_delta_offer=('delta_proposer_pct','mean'),
    mean_delta_accept=('delta_accept_rate','mean'),
    mean_delta_compound=('delta_compound_payoff','mean'),
    mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
).add_suffix('_LvL')

lvrb_agg = df_stats[df_stats['paradigm']=='LLM-vs-RuleBased'].groupby(['dimension','role']).agg(
    mean_delta_offer=('delta_proposer_pct','mean'),
    mean_delta_accept=('delta_accept_rate','mean'),
    mean_delta_compound=('delta_compound_payoff','mean'),
    mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
).add_suffix('_LvRB')

cross = lvl_agg.join(lvrb_agg, how='inner').reset_index()

print('=== Cross-Paradigm Comparison (matched on dimension × role) ===')
display(cross.round(3).sort_values('mean_delta_compound_LvRB', ascending=False))

In [ ]:
if not cross.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    met_pairs = [
        ('mean_delta_offer_LvL', 'mean_delta_offer_LvRB', 'Δ Offer (%)'),
        ('mean_delta_accept_LvL', 'mean_delta_accept_LvRB', 'Δ Accept Rate'),
        ('mean_delta_compound_LvL', 'mean_delta_compound_LvRB', 'Δ Compound Payoff'),
    ]
    for ax, (xm, ym, label) in zip(axes, met_pairs):
        sub = cross.dropna(subset=[xm, ym])
        colors_roles = sub['role'].map({'proposer': PALETTE[2], 'responder': PALETTE[3]})
        ax.scatter(sub[xm], sub[ym], c=colors_roles, s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
        lim = max(sub[[xm, ym]].abs().max().max() * 1.1, 1)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='y=x (same effect)')
        ax.axhline(0, color='grey', lw=0.6, linestyle=':')
        ax.axvline(0, color='grey', lw=0.6, linestyle=':')
        ax.set_xlabel(f'LLM-vs-LLM: {label}')
        ax.set_ylabel(f'LLM-vs-RuleBased: {label}')
        ax.set_title(label)
        # Label outliers
        for _, row in sub.iterrows():
            if abs(row[xm]) > lim*0.5 or abs(row[ym]) > lim*0.5:
                ax.annotate(f"{row['dimension']}\n({row['role'][0]})",
                            (row[xm], row[ym]), fontsize=6, ha='center', va='bottom')
        handles = [mpatches.Patch(color=PALETTE[2], label='proposer'),
                   mpatches.Patch(color=PALETTE[3], label='responder')]
        ax.legend(handles=handles, fontsize=8)

    plt.suptitle('LLM-vs-LLM vs LLM-vs-RuleBased:\nAre Effects Consistent Across Evaluation Paradigms?',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Correlation test
    for _, (xm, ym, label) in enumerate(met_pairs):
        sub = cross.dropna(subset=[xm, ym])
        if len(sub) >= 3:
            r, p = stats.pearsonr(sub[xm], sub[ym])
            print(f'{label}: r={r:.3f}, p={p:.4f}, n={len(sub)}')
else:
    print('No overlapping (dimension, role) pairs between paradigms — cross-paradigm scatter not applicable.')

---
## 13 · Full Statistical Summary Tables

In [ ]:
# One clean, publication-ready table for each paradigm
pub_cols = ['dimension', 'role', 'layer_num', 'best_alpha',
            'bl_accept_rate', 'fin_accept_rate', 'delta_accept_rate',
            'bl_mean_proposer_pct', 'fin_mean_proposer_pct', 'delta_proposer_pct',
            'bl_compound_payoff', 'fin_compound_payoff', 'delta_compound_payoff',
            'offer_cohens_d', 'offer_p', 'offer_sig_bh',
            'accept_p', 'accept_sig_bh']
pub_rename = {
    'dimension': 'Dimension', 'role': 'Role', 'layer_num': 'Layer', 'best_alpha': 'Best α',
    'bl_accept_rate': 'Acc (BL)', 'fin_accept_rate': 'Acc (S)', 'delta_accept_rate': 'Δ Acc',
    'bl_mean_proposer_pct': 'Offer (BL)', 'fin_mean_proposer_pct': 'Offer (S)', 'delta_proposer_pct': 'Δ Offer',
    'bl_compound_payoff': 'Cpd (BL)', 'fin_compound_payoff': 'Cpd (S)', 'delta_compound_payoff': 'Δ Cpd',
    'offer_cohens_d': "d (offer)", 'offer_p': 'p (offer)', 'offer_sig_bh': 'Sig* (offer)',
    'accept_p': 'p (accept)', 'accept_sig_bh': 'Sig* (accept)'
}

for par in df_stats['paradigm'].unique():
    sub = df_stats[df_stats['paradigm'] == par].copy()
    available = [c for c in pub_cols if c in sub.columns]
    sub = sub[available].rename(columns=pub_rename)
    sub = sub.sort_values(['Dimension', 'Role', 'Layer'])
    print(f'\n═══ {par}: Full Results Table ═══')
    display(sub.round(3))

---
## 14 · Acceptance Curve Analysis

We visualise how acceptance rate varies as a function of the *proposer's offer* for baseline vs steered games.
A non-monotonic acceptance curve indicates RLHF fairness enforcement (rejecting overly greedy AND overly generous offers).

In [ ]:
# Bin offers into deciles
df_games['offer_bin'] = pd.cut(df_games['proposer_pct'], bins=np.arange(0, 101, 10),
                               include_lowest=True)
df_games['offer_bin_mid'] = df_games['offer_bin'].apply(
    lambda x: x.mid if not pd.isna(x) else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax_idx, par in enumerate(['LLM-vs-LLM', 'LLM-vs-RuleBased']):
    sub = df_games[df_games['paradigm'] == par]
    ax  = axes[ax_idx]
    for tag, grp in sub.groupby('alpha_tag'):
        if tag == 'final':
            continue
        agg = grp.groupby('offer_bin_mid')['agreed'].agg(['mean', 'count']).reset_index()
        agg = agg[agg['count'] >= 5]  # filter low-count bins
        alpha = 0.0 if tag == 'baseline' else float(str(tag).replace('alpha',''))
        lw = 2.5 if tag == 'baseline' else 1.2
        ls = '-'  if tag == 'baseline' else '--'
        col = 'black' if tag == 'baseline' else None
        ax.plot(agg['offer_bin_mid'], agg['mean']*100, lw=lw, ls=ls,
                color=col, label=f'α={alpha}', alpha=0.8 if tag != 'baseline' else 1.0)

    ax.axvline(50, color='red', lw=1, linestyle=':', label='50% split')
    ax.set_xlabel('Proposer offer (% of pool)')
    ax.set_ylabel('Acceptance rate (%)')
    ax.set_title(f'{par}')
    ax.set_ylim(-5, 110)
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Acceptance Curve: Rate vs Offer Level (Baseline vs All Alphas)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15 · Parse Error and Data Quality Audit

In [ ]:
parse_err = (
    df_games
    .groupby(['paradigm', 'alpha_tag'])
    .agg(n=('parse_error','count'), errors=('parse_error','sum'))
    .assign(error_rate=lambda x: x['errors'] / x['n'])
    .reset_index()
    .sort_values('error_rate', ascending=False)
)
print('=== Parse Error Rates ===')
display(parse_err[parse_err['errors'] > 0])

total_errors = parse_err['errors'].sum()
total_games  = parse_err['n'].sum()
print(f'\nOverall parse error rate: {total_errors}/{total_games} = {total_errors/total_games:.3%}')

# Flag any configs with >10% errors
per_config_err = (
    df_stats[['paradigm','experiment','layer','role','dimension',
              'bl_n_parse_errors','fin_n_parse_errors']]
    .assign(total_errors=lambda x: x['bl_n_parse_errors'] + x['fin_n_parse_errors'])
    .query('total_errors > 5')
    .sort_values('total_errors', ascending=False)
)
if not per_config_err.empty:
    print('\nConfigs with high parse errors (>5 across baseline+final):')
    display(per_config_err)
else:
    print('\nNo configs with >5 parse errors. Data quality: GOOD.')

---
## 16 · Summary: Key Findings

In [ ]:
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║               SUMMARY: KEY EMPIRICAL FINDINGS                       ║')
print('╚══════════════════════════════════════════════════════════════════════╝')

for par in ['LLM-vs-LLM', 'LLM-vs-RuleBased']:
    sub = df_stats[df_stats['paradigm'] == par]
    if sub.empty:
        continue
    n_total = len(sub)
    n_sig_offer  = sub['offer_sig_bh'].sum()
    n_sig_accept = sub['accept_sig_bh'].sum()
    best_cpd = sub.loc[sub['delta_compound_payoff'].idxmax()]
    worst_cpd= sub.loc[sub['delta_compound_payoff'].idxmin()]
    
    print(f'\n── {par} ──')
    print(f'  Configurations:   {n_total}')
    print(f'  Sig offer shift (BH): {n_sig_offer}/{n_total} = {n_sig_offer/n_total:.1%}')
    print(f'  Sig accept shift (BH): {n_sig_accept}/{n_total} = {n_sig_accept/n_total:.1%}')
    print(f'  Mean Δ offer:    {sub["delta_proposer_pct"].mean():+.2f}%')
    print(f'  Mean Δ accept:   {sub["delta_accept_rate"].mean()*100:+.2f}pp')
    print(f'  Mean Δ compound: {sub["delta_compound_payoff"].mean():+.2f}pp')
    print(f'  Best config (Δ compound):  {best_cpd["dimension"]:15s} | {best_cpd["role"]:10s} | L{best_cpd["layer_num"]} α={best_cpd["best_alpha"]} → Δ={best_cpd["delta_compound_payoff"]:+.1f}pp')
    print(f'  Worst config (Δ compound): {worst_cpd["dimension"]:15s} | {worst_cpd["role"]:10s} | L{worst_cpd["layer_num"]} α={worst_cpd["best_alpha"]} → Δ={worst_cpd["delta_compound_payoff"]:+.1f}pp')

print()
print('  NOTE: Compound payoff = accept_rate × offer_pct.')
print('  Positive Δ compound = steering improves expected earnings.')
print('  BH-FDR correction applied across all configs within each paradigm.')

In [ ]:
# Final summary figure — radar / spider chart per paradigm
from matplotlib.patches import FancyArrowPatch

dim_means = (
    df_stats
    .groupby(['paradigm', 'dimension'])
    ['delta_compound_payoff']
    .mean()
    .unstack('paradigm')
)

dims_r = dim_means.index.tolist()
N = len(dims_r)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for i, par in enumerate(dim_means.columns):
    vals = dim_means[par].fillna(0).tolist()
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', color=PALETTE[i], linewidth=2, label=par)
    ax.fill(angles, vals, color=PALETTE[i], alpha=0.15)

ax.set_thetagrids(np.degrees(angles[:-1]), dims_r, fontsize=9)
ax.set_title('Compound Payoff Δ by Dimension\n(mean across layers and roles)',
             fontsize=11, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.axhline(0, color='grey', lw=0.8, linestyle='--')
plt.tight_layout()
plt.show()

---
## Appendix A · Full Per-Alpha Detail Table

In [ ]:
# Rebuild per-alpha summary from games data
per_alpha = (
    df_games[df_games['alpha_tag'] != 'final']
    .groupby(['paradigm','experiment','layer','role','dimension','alpha_num'])
    .agg(
        n=('agreed','count'),
        accept_rate=('agreed','mean'),
        mean_offer=('proposer_pct','mean'),
        std_offer=('proposer_pct','std'),
        mean_compound=('compound_payoff_pct','mean'),
        parse_errors=('parse_error','sum'),
    )
    .round(3)
    .reset_index()
    .sort_values(['paradigm','dimension','role','layer','alpha_num'])
)
print('=== Per-Alpha Aggregated Results (Appendix) ===')
display(per_alpha)

---
## Appendix B · Experiment-Level Metadata

In [ ]:
exp_meta = (
    df_stats
    .groupby(['paradigm', 'experiment'])
    .agg(
        layers=('layer_num', lambda x: sorted(x.unique())),
        roles=('role', lambda x: sorted(x.unique())),
        dimensions=('dimension', lambda x: sorted(x.unique())),
        n_configs=('dimension','count'),
        mean_delta_compound=('delta_compound_payoff','mean'),
    )
    .reset_index()
)
print('=== Experiment Metadata ===')
display(exp_meta)